# RSNA Knee Abnormality Detection — Complete Solution
> **Competition**: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection  
> **Task**: Detect knee abnormalities from multimodal data (MRI DICOM images + multilingual radiology reports)  
> **Metric**: Macro ROC-AUC  
> **Output**: `/kaggle/working/submission.csv`

In [ ]:
# ── 1. Install non-default packages ─────────────────────────────────────
import subprocess, sys

pkgs = [
    'timm==0.9.12',
    'albumentations==1.3.1',
    'pydicom',
    'transformers==4.40.0',
    'sentencepiece',
    'accelerate',
]
for pkg in pkgs:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        check=False, capture_output=True
    )
print('Installation complete.')


In [ ]:
# ── 2. Imports & global config ───────────────────────────────────────────
import os, gc, glob, random, warnings, math
import numpy as np
import pandas as pd
import cv2
import pydicom
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
from transformers import AutoTokenizer, AutoModel, AutoConfig

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# ── Paths ─────────────────────────────────────────────────────────────────
BASE_DIR   = Path('/kaggle/input/rsna-knee-abnormality-detection')
TRAIN_DIR  = BASE_DIR / 'train'
TEST_DIR   = BASE_DIR / 'test'
OUT_DIR    = Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Offline model weight paths ─────────────────────────────────────────────
# HOW TO SET UP (do this ONCE before submitting):
#   1. Go to kaggle.com/models → search "xlm-roberta-base" → Add to notebook
#      → path becomes /kaggle/input/xlm-roberta-base/  (contains config.json etc.)
#   2. Go to kaggle.com/models → search "efficientnet-b4" under timm →
#      path becomes /kaggle/input/timm-efficientnet-b4/ (contains *.pth)
#   If paths are empty strings (""), models load from HuggingFace/timm cache
#   (works in interactive session with internet, fails in submission without).
IMG_WEIGHTS_PATH = '/kaggle/input/timm-efficientnet-b4'   # local timm checkpoint dir
TXT_WEIGHTS_PATH = '/kaggle/input/xlm-roberta-base'       # local HuggingFace model dir

def _resolve_model_path(local_path: str, hub_name: str) -> str:
    """Return local path if it exists with model files, else hub name."""
    p = Path(local_path)
    if p.exists() and any(p.iterdir()):
        print(f'  [offline] Loading from {local_path}')
        return str(local_path)
    print(f'  [online ] No local weights at {local_path!r} — using hub: {hub_name}')
    return hub_name

# ── Hyper-parameters ──────────────────────────────────────────────────────
CFG = dict(
    img_size     = 224,
    batch_size   = 8,
    n_epochs     = 5,
    lr           = 2e-4,
    weight_decay = 1e-2,
    n_folds      = 5,
    device       = 'cuda' if torch.cuda.is_available() else 'cpu',
    img_model    = 'efficientnet_b4',
    txt_model    = 'xlm-roberta-base',
    max_txt_len  = 256,
    n_slices     = 5,
    mixed_prec   = True,
    num_workers  = 2,
)

print('Device :', CFG['device'])
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# ── 3. Load metadata & discover columns ──────────────────────────────────
def safe_read_csv(path):
    """Read a CSV if it exists, else return empty DataFrame."""
    if Path(path).exists():
        return pd.read_csv(path)
    print(f'WARNING: {path} not found')
    return pd.DataFrame()

train_df   = safe_read_csv(BASE_DIR / 'train.csv')
test_df    = safe_read_csv(BASE_DIR / 'test.csv')
sample_sub = safe_read_csv(BASE_DIR / 'sample_submission.csv')

print('train_df  :', train_df.shape)
print('test_df   :', test_df.shape)
print('sample_sub:', sample_sub.shape)
print()
print('--- train columns ---')
print(train_df.columns.tolist())
print()
print('--- sample_submission columns ---')
print(sample_sub.columns.tolist())
print()
print(train_df.head(3))

# ── Detect study-ID column ────────────────────────────────────────────────
ID_COL_CANDIDATES = ['study_id', 'studyuid', 'id', 'row_id', 'StudyInstanceUID']
ID_COL = next(
    (c for c in ID_COL_CANDIDATES if c in sample_sub.columns),
    sample_sub.columns[0]   # fallback: first column
)
print('ID column detected:', ID_COL)

# ── Detect label columns: everything in sample_sub except the ID ──────────
LABEL_COLS = [c for c in sample_sub.columns if c != ID_COL]
print('Label columns:', LABEL_COLS)

# ── Detect text / report column in train_df ───────────────────────────────
TEXT_COL = next(
    (c for c in train_df.columns
     if any(k in c.lower() for k in ['report', 'text', 'finding', 'impression'])),
    None
)
print('Text column:', TEXT_COL)

# Make sure test_df also has TEXT_COL (may be empty)
if TEXT_COL and TEXT_COL not in test_df.columns:
    test_df[TEXT_COL] = ''

N_CLASSES = len(LABEL_COLS)
print('Number of output classes:', N_CLASSES)


In [ ]:
# ── 4. Exploratory data analysis ─────────────────────────────────────────
if len(train_df) > 0 and LABEL_COLS:
    fig, axes = plt.subplots(1, min(len(LABEL_COLS), 4),
                             figsize=(4 * min(len(LABEL_COLS), 4), 4))
    if len(LABEL_COLS) == 1:
        axes = [axes]
    for ax, col in zip(axes, LABEL_COLS[:4]):
        if col in train_df.columns:
            vc = train_df[col].value_counts()
            ax.bar(vc.index.astype(str), vc.values, color=['#4C72B0', '#DD8452'])
            ax.set_title(col)
            ax.set_xlabel('Label')
            ax.set_ylabel('Count')
    plt.suptitle('Label distribution (train)', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    print('Label statistics:')
    print(train_df[LABEL_COLS].describe())
else:
    print('Skipping EDA — empty train_df or no label columns detected')


In [ ]:
# ── 5. DICOM & image utilities ───────────────────────────────────────────

def get_dicom_paths(study_id, split='train'):
    """Return sorted list of DICOM paths for a study."""
    root = TRAIN_DIR if split == 'train' else TEST_DIR
    pattern = str(root / str(study_id) / '**' / '*.dcm')
    paths = sorted(glob.glob(pattern, recursive=True))
    if not paths:
        # Some competitions store files directly under study folder
        pattern2 = str(root / str(study_id) / '*.dcm')
        paths = sorted(glob.glob(pattern2))
    return paths


def dicom_to_uint8(dcm):
    """Convert a pydicom dataset to a uint8 numpy array."""
    img = dcm.pixel_array.astype(np.float32)
    # Apply slope / intercept
    slope = float(getattr(dcm, 'RescaleSlope', 1))
    intercept = float(getattr(dcm, 'RescaleIntercept', 0))
    img = img * slope + intercept
    # Normalize to 0-255
    lo, hi = img.min(), img.max()
    if hi > lo:
        img = (img - lo) / (hi - lo) * 255.0
    return img.astype(np.uint8)


def load_slice(path, size):
    """Load one DICOM slice and return RGB numpy array (H, W, 3)."""
    try:
        dcm = pydicom.dcmread(path)
        img = dicom_to_uint8(dcm)
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=-1)
        elif img.ndim == 3 and img.shape[-1] != 3:
            img = img[..., :3]
        img = cv2.resize(img, (size, size))
        return img
    except Exception as e:
        return np.zeros((size, size, 3), dtype=np.uint8)


# Quick sanity check
if len(train_df) > 0:
    sid = train_df[ID_COL].iloc[0]
    paths = get_dicom_paths(sid, 'train')
    print(f'Study {sid}: {len(paths)} DICOM files')
    if paths:
        img = load_slice(paths[len(paths)//2], CFG['img_size'])
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(f'Sample slice — study {sid}')
        plt.axis('off')
        plt.show()
    else:
        print('No DICOM files found for this study — check directory structure.')


In [ ]:
# ── 6. Augmentations ─────────────────────────────────────────────────────

def get_transforms(mode='train', size=None):
    size = size or CFG['img_size']
    if mode == 'train':
        return A.Compose([
            A.RandomResizedCrop(size, size, scale=(0.8, 1.0), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5
            ),
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0)),
                A.GaussianBlur(blur_limit=(3, 5)),
                A.MotionBlur(blur_limit=3),
            ], p=0.3),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.4),
            A.CoarseDropout(
                max_holes=8, max_height=16, max_width=16,
                fill_value=0, p=0.3
            ),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            ),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(size, size),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            ),
            ToTensorV2(),
        ])

print('Augmentation pipelines defined.')


In [ ]:
# ── 7. Tokenizer (singleton, loaded once) ────────────────────────────────
_TOKENIZER = None

def get_tokenizer():
    global _TOKENIZER
    if _TOKENIZER is None:
        path = _resolve_model_path(TXT_WEIGHTS_PATH, CFG['txt_model'])
        try:
            _TOKENIZER = AutoTokenizer.from_pretrained(path)
            print(f'Tokenizer loaded: {path}')
        except Exception as e:
            raise RuntimeError(
                f'Cannot load tokenizer from {path!r}.\n'
                f'  Error: {e}\n'
                f'  FIX: Attach the "{CFG["txt_model"]}" model as a Kaggle dataset\n'
                f'       (kaggle.com/models → search xlm-roberta-base → Add to notebook)'
            )
    return _TOKENIZER

_ = get_tokenizer()


In [ ]:
# ── 8. Dataset ───────────────────────────────────────────────────────────

class KneeDataset(Dataset):
    """
    One sample = one study.
    Returns fused image tensor (mean of N middle slices) + tokenised report.
    """
    def __init__(self, df, split='train'):
        self.df    = df.reset_index(drop=True)
        self.split = split
        self.tfms  = get_transforms(split)
        self.tok   = get_tokenizer()

    def __len__(self):
        return len(self.df)

    def _load_image_tensor(self, study_id):
        paths = get_dicom_paths(str(study_id), self.split)
        if not paths:
            return torch.zeros(3, CFG['img_size'], CFG['img_size'])

        # Select N slices centred around the middle
        n = CFG['n_slices']
        mid = len(paths) // 2
        half = n // 2
        selected = paths[max(0, mid - half): mid + half + 1][:n]

        tensors = []
        for p in selected:
            arr = load_slice(p, CFG['img_size'])
            try:
                t = self.tfms(image=arr)['image']   # (3, H, W)
                tensors.append(t)
            except Exception:
                pass

        if not tensors:
            return torch.zeros(3, CFG['img_size'], CFG['img_size'])
        return torch.stack(tensors).mean(0)   # average slices

    def _encode_text(self, text):
        if not isinstance(text, str) or not text.strip():
            text = 'no report available'
        enc = self.tok(
            text,
            max_length=CFG['max_txt_len'],
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row[ID_COL]

        img = self._load_image_tensor(study_id)

        text = str(row[TEXT_COL]) if TEXT_COL and TEXT_COL in row.index else ''
        input_ids, attn_mask = self._encode_text(text)

        item = {
            'image'     : img,
            'input_ids' : input_ids,
            'attn_mask' : attn_mask,
            'study_id'  : str(study_id),
        }

        if self.split != 'test' and LABEL_COLS:
            lbl = row[LABEL_COLS].values.astype(np.float32)
            item['labels'] = torch.tensor(lbl, dtype=torch.float32)

        return item


# Quick dataset test
if len(train_df) > 0:
    _ds = KneeDataset(train_df.head(2), split='train')
    _s  = _ds[0]
    print('Image shape  :', _s['image'].shape)
    print('input_ids    :', _s['input_ids'].shape)
    print('attn_mask    :', _s['attn_mask'].shape)
    if 'labels' in _s:
        print('labels       :', _s['labels'])
    del _ds, _s; gc.collect()


In [ ]:
# ── 9. Multimodal model ──────────────────────────────────────────────────

class ImageEncoder(nn.Module):
    def __init__(self, name=CFG['img_model']):
        super().__init__()
        local = Path(IMG_WEIGHTS_PATH)

        # Look for a .pth/.bin checkpoint inside the local dataset folder
        ckpt_files = list(local.glob('*.pth')) + list(local.glob('*.bin'))                      if local.exists() else []

        if ckpt_files:
            # Step 1: Build architecture (no download)
            self.backbone = timm.create_model(
                name, pretrained=False, num_classes=0, global_pool='avg'
            )
            # Step 2: Load local weights
            state = torch.load(ckpt_files[0], map_location='cpu')
            # Handle nested state dicts (some checkpoints wrap under 'model' key)
            if isinstance(state, dict) and 'model' in state:
                state = state['model']
            missing, unexpected = self.backbone.load_state_dict(state, strict=False)
            print(f'ImageEncoder loaded from {ckpt_files[0].name}  '
                  f'(missing={len(missing)}, unexpected={len(unexpected)})')
        else:
            # Internet session (interactive/non-submission): download normally
            try:
                self.backbone = timm.create_model(
                    name, pretrained=True, num_classes=0, global_pool='avg'
                )
                print(f'ImageEncoder: downloaded pretrained {name} from timm hub')
            except Exception as e:
                raise RuntimeError(
                    f'Cannot load image encoder weights.\n'
                    f'  Error: {e}\n'
                    f'  FIX: Attach a timm efficientnet-b4 checkpoint dataset at\n'
                    f'       {IMG_WEIGHTS_PATH!r}'
                )

        self.out_dim = self.backbone.num_features

    def forward(self, x):
        return self.backbone(x)   # (B, out_dim)


class TextEncoder(nn.Module):
    def __init__(self, name=CFG['txt_model']):
        super().__init__()
        path = _resolve_model_path(TXT_WEIGHTS_PATH, name)
        try:
            self.encoder = AutoModel.from_pretrained(path)
            print(f'TextEncoder loaded from: {path}')
        except Exception as e:
            raise RuntimeError(
                f'Cannot load text encoder from {path!r}.\n'
                f'  Error: {e}\n'
                f'  FIX: Attach the "{name}" model as a Kaggle dataset\n'
                f'       (kaggle.com/models → search xlm-roberta-base → Add to notebook)\n'
                f'  OR:  Set TXT_WEIGHTS_PATH to a valid local directory.'
            )
        self.out_dim = self.encoder.config.hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return out.last_hidden_state[:, 0, :]   # CLS token (B, hidden)


class MultimodalKneeModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.img_enc  = ImageEncoder()
        self.txt_enc  = TextEncoder()
        dim = self.img_enc.out_dim + self.txt_enc.out_dim
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(0.3),
            nn.Linear(dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, n_classes),
        )

    def forward(self, image, input_ids, attn_mask):
        img_feat  = self.img_enc(image)
        txt_feat  = self.txt_enc(input_ids, attn_mask)
        fused     = torch.cat([img_feat, txt_feat], dim=-1)
        return self.head(fused)


# Parameter count
_m = MultimodalKneeModel(n_classes=max(N_CLASSES, 1))
total_p = sum(p.numel() for p in _m.parameters()) / 1e6
trainable_p = sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6
print(f'Total params     : {total_p:.1f}M')
print(f'Trainable params : {trainable_p:.1f}M')
del _m; gc.collect()


In [ ]:
# ── 10. Training & validation helpers ────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    for step, batch in enumerate(loader):
        img   = batch['image'].to(device)
        ids   = batch['input_ids'].to(device)
        mask  = batch['attn_mask'].to(device)
        lbls  = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CFG['mixed_prec']):
            logits = model(img, ids, mask)
            loss   = criterion(logits, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        if (step + 1) % 20 == 0:
            print(f'  step {step+1}/{len(loader)}  loss={running_loss/(step+1):.4f}',
                  end='\r')

    return running_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in loader:
        img   = batch['image'].to(device)
        ids   = batch['input_ids'].to(device)
        mask  = batch['attn_mask'].to(device)
        lbls  = batch['labels'].to(device)

        with autocast(enabled=CFG['mixed_prec']):
            logits = model(img, ids, mask)
            loss   = criterion(logits, lbls)

        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(lbls.cpu().numpy())

    preds  = np.vstack(all_preds)
    labels = np.vstack(all_labels)

    # Macro ROC-AUC (skip classes with only one unique label)
    aucs = []
    for i in range(labels.shape[1]):
        if len(np.unique(labels[:, i])) > 1:
            aucs.append(roc_auc_score(labels[:, i], preds[:, i]))
    macro_auc = float(np.mean(aucs)) if aucs else 0.0

    return total_loss / len(loader), macro_auc, preds


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_preds = []
    for batch in loader:
        img  = batch['image'].to(device)
        ids  = batch['input_ids'].to(device)
        mask = batch['attn_mask'].to(device)
        with autocast(enabled=CFG['mixed_prec']):
            logits = model(img, ids, mask)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(all_preds)


print('Training helpers defined.')


In [ ]:
# ── 11. K-Fold cross-validation training ─────────────────────────────────

oof_preds = np.zeros((len(train_df), max(N_CLASSES, 1)))
fold_aucs = []

# Stratify on first label; use zeros if no label found
if LABEL_COLS and LABEL_COLS[0] in train_df.columns:
    strat_col = train_df[LABEL_COLS[0]].fillna(0).astype(int)
else:
    strat_col = pd.Series(np.zeros(len(train_df), dtype=int))

skf = StratifiedKFold(
    n_splits=CFG['n_folds'], shuffle=True, random_state=SEED
)

for fold, (tr_idx, vl_idx) in enumerate(
    skf.split(np.zeros(len(train_df)), strat_col)
):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold} | train={len(tr_idx)} val={len(vl_idx)}')
    print(f'{"="*55}')

    tr_ds = KneeDataset(train_df.iloc[tr_idx], split='train')
    vl_ds = KneeDataset(train_df.iloc[vl_idx], split='val')

    tr_loader = DataLoader(
        tr_ds,
        batch_size=CFG['batch_size'],
        shuffle=True,
        num_workers=CFG['num_workers'],
        pin_memory=True,
        drop_last=True,
    )
    vl_loader = DataLoader(
        vl_ds,
        batch_size=CFG['batch_size'] * 2,
        shuffle=False,
        num_workers=CFG['num_workers'],
        pin_memory=True,
    )

    model = MultimodalKneeModel(n_classes=max(N_CLASSES, 1))
    model = model.to(CFG['device'])

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG['lr'],
        weight_decay=CFG['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['n_epochs'], eta_min=1e-6
    )
    scaler = GradScaler(enabled=CFG['mixed_prec'])

    best_auc   = -1.0
    best_preds = None
    ckpt_path  = OUT_DIR / f'best_fold{fold}.pth'

    for epoch in range(1, CFG['n_epochs'] + 1):
        tr_loss = train_one_epoch(
            model, tr_loader, optimizer, criterion, scaler, CFG['device']
        )
        vl_loss, vl_auc, vl_preds = evaluate(
            model, vl_loader, criterion, CFG['device']
        )
        scheduler.step()

        print(f'  Epoch {epoch}/{CFG["n_epochs"]} '
              f'| tr_loss={tr_loss:.4f} '
              f'| vl_loss={vl_loss:.4f} '
              f'| vl_AUC={vl_auc:.4f}')

        if vl_auc > best_auc:
            best_auc   = vl_auc
            best_preds = vl_preds.copy()
            torch.save(model.state_dict(), ckpt_path)
            print(f'    ✓ Best AUC updated → {best_auc:.4f}')

    oof_preds[vl_idx] = best_preds if best_preds is not None else vl_preds
    fold_aucs.append(best_auc)

    del model, tr_ds, vl_ds, tr_loader, vl_loader
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*55}')
print(f'Fold AUCs : {[round(a,4) for a in fold_aucs]}')
print(f'Mean AUC  : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')
print(f'{"="*55}')


In [ ]:
# ── 12. OOF (out-of-fold) evaluation ────────────────────────────────────
if LABEL_COLS:
    oof_aucs = []
    for i, col in enumerate(LABEL_COLS):
        if col in train_df.columns:
            true = train_df[col].values
            pred = oof_preds[:, i]
            if len(np.unique(true)) > 1:
                auc = roc_auc_score(true, pred)
                oof_aucs.append(auc)
                print(f'  {col:40s}: AUC = {auc:.4f}')
    print(f'\nOverall OOF Macro AUC : {np.mean(oof_aucs):.4f}')

    # Plot OOF score distribution
    fig, axes = plt.subplots(
        1, min(len(LABEL_COLS), 4), figsize=(4 * min(len(LABEL_COLS), 4), 3)
    )
    if len(LABEL_COLS) == 1:
        axes = [axes]
    for ax, col in zip(axes, LABEL_COLS[:4]):
        if col in train_df.columns:
            ax.hist(oof_preds[:, LABEL_COLS.index(col)], bins=40,
                    color='steelblue', edgecolor='white')
            ax.set_title(f'OOF predictions — {col}')
            ax.set_xlabel('Predicted probability')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 13. Test inference (ensemble across folds) ───────────────────────────
test_preds = np.zeros((len(test_df), max(N_CLASSES, 1)))
n_loaded   = 0

test_ds     = KneeDataset(test_df, split='test')
test_loader = DataLoader(
    test_ds,
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
)

for fold in range(CFG['n_folds']):
    ckpt_path = OUT_DIR / f'best_fold{fold}.pth'
    if not ckpt_path.exists():
        print(f'  Fold {fold}: checkpoint not found at {ckpt_path} — skipping.')
        continue
    model = MultimodalKneeModel(n_classes=max(N_CLASSES, 1)).to(CFG['device'])
    model.load_state_dict(
        torch.load(ckpt_path, map_location=CFG['device'])
    )
    fold_preds  = predict(model, test_loader, CFG['device'])
    test_preds += fold_preds
    n_loaded   += 1
    print(f'  Fold {fold} inference done  '
          f'(pred mean={fold_preds.mean():.4f}, std={fold_preds.std():.4f})')
    del model; gc.collect(); torch.cuda.empty_cache()

# ── Hard guard: no checkpoints = no submission ────────────────────────────
if n_loaded == 0:
    raise RuntimeError(
        'No fold checkpoints were found in /kaggle/working/!\n'
        '  This means training Cell 11 did not complete or save any models.\n'
        '  Run Cell 11 fully before running inference.\n'
        '  Submitting zeros/constants produces AUC = 0.500 (random baseline).'
    )

test_preds /= n_loaded           # average over folds
test_preds  = np.clip(test_preds, 0.0, 1.0)

# ── Constant-prediction guard ──────────────────────────────────────────────
pred_std = test_preds.std()
pred_mean = test_preds.mean()
print(f'\nEnsemble predictions  mean={pred_mean:.4f}  std={pred_std:.6f}')
print(f'Folds loaded: {n_loaded}/{CFG["n_folds"]}')

if pred_std < 1e-4:
    raise RuntimeError(
        f'CONSTANT PREDICTIONS DETECTED (std={pred_std:.2e}, mean={pred_mean:.4f}).\n'
        f'  This will produce AUC = 0.500 (random baseline) on the leaderboard.\n'
        f'  Most likely causes:\n'
        f'    1. Pretrained image weights not loaded (IMG_WEIGHTS_PATH not set or empty dir)\n'
        f'       → sigmoid(0) = 0.5 for all inputs\n'
        f'    2. Pretrained text weights not loaded (TXT_WEIGHTS_PATH not set or empty dir)\n'
        f'       → same sigmoid(0) = 0.5 effect\n'
        f'    3. Training converged to a degenerate solution (check training loss curve)\n'
        f'  FIX: Attach xlm-roberta-base and timm-efficientnet-b4 as Kaggle datasets,\n'
        f'       then set IMG_WEIGHTS_PATH and TXT_WEIGHTS_PATH in Cell 2.'
    )

print(f'Test predictions shape : {test_preds.shape}')
print(f'Pred min/max           : {test_preds.min():.4f} / {test_preds.max():.4f}')


In [ ]:
# ── 14. Build & validate submission.csv ──────────────────────────────────

# Re-read sample to get exact column order
sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')

submission = pd.DataFrame()
submission[ID_COL] = test_df[ID_COL].values

for i, col in enumerate(LABEL_COLS):
    submission[col] = test_preds[:, i]

# Reorder columns to exactly match sample_submission
submission = submission[sample_sub.columns]

# ── Validation checks ────────────────────────────────────────────────────
assert len(submission) == len(sample_sub), (
    f'Row count mismatch: got {len(submission)}, expected {len(sample_sub)}'
)
assert list(submission.columns) == list(sample_sub.columns), (
    f'Column mismatch:\n  got      {list(submission.columns)}'
    f'\n  expected {list(sample_sub.columns)}'
)
for col in LABEL_COLS:
    assert submission[col].notna().all(), f'NaN values found in column: {col}'
    assert (submission[col] >= 0.0).all() and (submission[col] <= 1.0).all(), (
        f'Predictions out of [0,1] in column: {col}'
    )
    # Warn if any single label column is constant (AUC = 0.5 for that target)
    col_std = submission[col].std()
    if col_std < 1e-4:
        print(f'WARNING: column {col!r} has std={col_std:.2e} (near-constant). '
              f'AUC for this label will be ~0.500.')

# ── Save ─────────────────────────────────────────────────────────────────
SUB_PATH = OUT_DIR / 'submission.csv'
submission.to_csv(SUB_PATH, index=False)

print('=' * 55)
print(f'  submission.csv saved → {SUB_PATH}')
print(f'  Shape  : {submission.shape}')
print(f'  Columns: {submission.columns.tolist()}')
print('=' * 55)
print(submission.head(10))
print()
print('Prediction statistics:')
print(submission[LABEL_COLS].describe().round(4))
